# Bula Clara - Assistente RAG de Bulas

Este notebook documenta a construcao de um assistente para consulta explicativa de bulas. A ideia e usar RAG: buscar trechos dos PDFs antes de pedir para a LLM gerar uma resposta.

**Aviso importante:** este projeto e informativo. Ele nao substitui medico, farmaceutico ou a bula oficial.

## 1. Objetivo do projeto

- Ler bulas em PDF.
- Dividir os textos em trechos menores.
- Criar embeddings locais, sem consumir tokens da LLM.
- Salvar os vetores no ChromaDB.
- Recuperar trechos relevantes para cada pergunta.
- Gerar uma resposta com Groq mostrando as fontes usadas.

In [ ]:
from pathlib import Path

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

## 2. Parametros principais

Aqui ficam as decisoes centrais do projeto. Se voce trocar os PDFs, normalmente so precisa atualizar `PDF_FILES`.

In [ ]:
PDF_FILES = [Path("dipirona.pdf"), Path("paracetamol.pdf")]
CHROMA_DIR = Path("chroma_bula_clara_local")
COLLECTION_NAME = "bula_clara_local"
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
LLM_MODEL = "llama-3.1-8b-instant"
RETRIEVER_K = 4

## 3. Carregamento das bulas

Cada pagina vira um documento. Tambem adicionamos o nome do medicamento como metadado para melhorar a explicacao das fontes.

In [ ]:
def infer_medicine_name(path: Path) -> str:
    return path.stem.replace("_", " ").replace("-", " ").title()


documents = []
for pdf_path in PDF_FILES:
    loaded_docs = PyPDFLoader(str(pdf_path)).load()
    medicine = infer_medicine_name(pdf_path)
    for doc in loaded_docs:
        doc.metadata["medicamento"] = medicine
        doc.metadata["source"] = pdf_path.name
    documents.extend(loaded_docs)

len(documents), documents[0].metadata

## 4. Divisao em chunks

A LLM nao recebe o PDF inteiro. O retriever busca pequenos trechos que combinam semanticamente com a pergunta.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=160,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(documents)
len(chunks), chunks[0].page_content[:500]

## 5. Metadados de categoria

Criamos uma classificacao simples por palavras-chave. Isso nao e um classificador perfeito, mas ajuda a deixar as fontes mais explicaveis.

In [ ]:
CATEGORY_KEYWORDS = {
    "identificacao": ["identificacao do medicamento", "apresentacoes", "composicao"],
    "indicacao": ["para que este medicamento e indicado", "indicacoes"],
    "como_funciona": ["como este medicamento funciona"],
    "contraindicacao": ["quando nao devo usar", "contraindicacoes", "contra-indicacoes"],
    "advertencias_precaucoes": ["o que devo saber antes", "advertencias", "precaucoes"],
    "interacoes": ["interacoes medicamentosas", "interacao medicamentosa"],
    "posologia_modo_uso": ["como devo usar", "posologia", "modo de usar"],
    "reacoes_adversas": ["quais os males", "reacoes adversas", "efeitos colaterais"],
    "armazenamento": ["onde como e por quanto tempo posso guardar", "conservacao"],
    "superdosagem": ["o que fazer se alguem usar uma quantidade maior", "superdose"],
}


def classify_category(text: str) -> str:
    normalized = text.lower()
    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(keyword in normalized for keyword in keywords):
            return category
    return "geral"


for chunk in chunks:
    chunk.metadata["categoria"] = classify_category(chunk.page_content)

chunks[0].metadata

## 6. Embeddings locais e ChromaDB

Os embeddings rodam localmente com Sentence Transformers. Isso evita custo por embedding na API da LLM.

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

if vectorstore._collection.count() == 0:
    vectorstore.add_documents(chunks)

vectorstore._collection.count()

## 7. Teste do retriever

Antes de usar a LLM, vale verificar se a busca esta trazendo trechos coerentes.

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": RETRIEVER_K})
docs = retriever.invoke("Quais sao as contraindicacoes da dipirona?")

for i, doc in enumerate(docs, start=1):
    print("---", i, doc.metadata)
    print(doc.page_content[:700].replace("\n", " "))

## 8. Prompt seguro para saude

O prompt limita o comportamento do modelo: responder somente com base no contexto e nao dar diagnostico ou orientacao individual.

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Voce e um assistente informativo de bulas. Responda somente com base no contexto. "
            "Nao de diagnostico, nao recomende iniciar, parar ou alterar medicamento. "
            "Se a informacao nao estiver no contexto, diga que nao encontrou.\n\nContexto:\n{context}",
        ),
        ("human", "{input}"),
    ]
)

## 9. Geracao com Groq

Cole sua chave apenas em tempo de execucao. Nao salve chaves no notebook, no GitHub ou no Hugging Face.

In [ ]:
import getpass

groq_api_key = getpass.getpass("Cole sua GROQ_API_KEY: ")
llm = ChatGroq(groq_api_key=groq_api_key, model=LLM_MODEL, temperature=0)

In [ ]:
question = "Quais sao as principais contraindicacoes da dipirona?"
docs = retriever.invoke(question)
context = "\n\n".join(doc.page_content for doc in docs)

messages = prompt.invoke({"input": question, "context": context})
response = llm.invoke(messages)
print(response.content)

## 10. Conclusao

Este projeto mostra uma arquitetura util para portfolio: documentos reais, embeddings locais, banco vetorial, LLM em cloud e fontes visiveis. O proximo passo natural e comparar modelos de embedding e medir quais perguntas recuperam melhor os trechos corretos.